## 1. Import Libraries

For today's Proof of Concept - **Deploy a Simple ML API** - we need libraries for training a
model, serializing it, building a REST API with FastAPI, and (lightly) tracking experiments
with MLflow.

In [1]:
import os
import json
import joblib
import pickle
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

print('Libraries ready')

Libraries ready


c:\Users\Mathew\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. Train a Simple ML Model

We'll train a small **Logistic Regression** classifier on the classic Iris dataset - the goal
today isn't the model itself (we covered that on Day 8), it's **everything that happens after
training**: saving the model, wrapping it in an API, containerizing it, and deploying it.

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))
print(f'Model trained. Test accuracy: {accuracy:.2f}')

Model trained. Test accuracy: 1.00


## 3. ML Model Serialization (pickle / joblib)

Before a model can be served by an API, it needs to be **serialized** - saved to disk in a
format that can be loaded back later without retraining. Two common options in Python:

- **pickle** - Python's built-in general-purpose serialization module.
- **joblib** - optimized for objects with large NumPy arrays (like most scikit-learn models);
  usually faster and produces smaller files for ML models.

Let's save our model both ways and confirm we can load it back and get identical predictions.

In [3]:
os.makedirs('artifacts', exist_ok=True)

# Save with pickle
with open('artifacts/model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save with joblib (preferred for scikit-learn models)
joblib.dump(model, 'artifacts/model.joblib')

print('Saved model with pickle ->', os.path.getsize('artifacts/model.pkl'), 'bytes')
print('Saved model with joblib ->', os.path.getsize('artifacts/model.joblib'), 'bytes')

Saved model with pickle -> 788 bytes
Saved model with joblib -> 975 bytes


In [4]:
# Load both versions back and confirm predictions match the original model
with open('artifacts/model.pkl', 'rb') as f:
    pickle_model = pickle.load(f)

joblib_model = joblib.load('artifacts/model.joblib')

sample = X_test[:3]
print('Original predictions :', model.predict(sample))
print('Pickle-loaded preds  :', pickle_model.predict(sample))
print('Joblib-loaded preds  :', joblib_model.predict(sample))

Original predictions : [1 0 2]
Pickle-loaded preds  : [1 0 2]
Joblib-loaded preds  : [1 0 2]


## 4. MLflow Overview

**MLflow** helps track ML experiments - logging parameters, metrics, and model artifacts so
you can compare runs and reproduce results later. It has four main components: **Tracking**,
**Projects**, **Models**, and a **Model Registry**. Here we use a local SQLite-backed **Tracking** store (no separate server
needed) to log this model's parameters, accuracy, and the model artifact itself.

In [5]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('iris-day10-demo')

with mlflow.start_run(run_name='logistic-regression-iris'):
    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('max_iter', 200)
    mlflow.log_metric('accuracy', accuracy)
    mlflow.sklearn.log_model(model, name='model')
    run_id = mlflow.active_run().info.run_id

print(f'MLflow run logged. Run ID: {run_id}')
print('Tracking data stored in ./mlflow.db (open with `mlflow ui --backend-store-uri sqlite:///mlflow.db`)')

2026/07/22 10:12:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/22 10:12:29 INFO mlflow.store.db.utils: Updating database tables
2026/07/22 10:13:03 INFO mlflow.tracking.fluent: Experiment with name 'iris-day10-demo' does not exist. Creating a new experiment.


MLflow run logged. Run ID: 01e4cb895db04bcbb69aa49b469f1feb
Tracking data stored in ./mlflow.db (open with `mlflow ui --backend-store-uri sqlite:///mlflow.db`)


## 5. FastAPI & REST API Creation

**FastAPI** is a modern Python web framework for building APIs quickly, with automatic request
validation (via **Pydantic**) and interactive docs (Swagger UI) generated for free.

A minimal ML-serving API needs at least:

- A **request schema** describing the expected input (our 4 iris measurements).
- A **`/predict` endpoint** that loads the model and returns a prediction.
- A **`/health` endpoint** so deployment tools can check the service is alive.

Below is the FastAPI app. In a real project this would live in its own `app.py` file (we also
write it to disk further down so it can be containerized), but we define it inline first so we
can test it directly inside the notebook.

In [6]:
class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

class PredictionResponse(BaseModel):
    predicted_class: str
    predicted_class_index: int
    confidence: float

app = FastAPI(title='Iris Prediction API', version='1.0.0')

# Load the serialized model once, at startup (not on every request)
serving_model = joblib.load('artifacts/model.joblib')

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.post('/predict', response_model=PredictionResponse)
def predict(request: IrisRequest):
    features = np.array([[request.sepal_length, request.sepal_width,
                           request.petal_length, request.petal_width]])
    prediction = serving_model.predict(features)[0]
    probabilities = serving_model.predict_proba(features)[0]

    return PredictionResponse(
        predicted_class=target_names[prediction],
        predicted_class_index=int(prediction),
        confidence=float(probabilities[prediction])
    )

print('FastAPI app created with /health and /predict endpoints')

FastAPI app created with /health and /predict endpoints


## 6. Testing the API (no server needed)

FastAPI's `TestClient` lets us call the API endpoints directly inside Python/the notebook,
exactly like a real HTTP client would - useful for quick testing and for automated tests in a
CI/CD pipeline (see the GitHub Actions section below).

In [7]:
client = TestClient(app)

health_response = client.get('/health')
print('GET /health ->', health_response.status_code, health_response.json())

sample_request = {
    'sepal_length': 5.1,
    'sepal_width': 3.5,
    'petal_length': 1.4,
    'petal_width': 0.2
}
predict_response = client.post('/predict', json=sample_request)
print('POST /predict ->', predict_response.status_code)
print(json.dumps(predict_response.json(), indent=2))

GET /health -> 200 {'status': 'ok'}
POST /predict -> 200
{
  "predicted_class": "setosa",
  "predicted_class_index": 0,
  "confidence": 0.9765532842480741
}


## 7. Writing the API to a Standalone `app.py`

To actually **run** the API with a real server (`uvicorn app:app --host 0.0.0.0 --port 8000`)
or containerize it with Docker, the app needs to live in its own file rather than a notebook
cell. The cell below writes an equivalent `app.py` to disk.

In [8]:
app_py_code = '''\
import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel

TARGET_NAMES = ["setosa", "versicolor", "virginica"]

app = FastAPI(title="Iris Prediction API", version="1.0.0")
model = joblib.load("artifacts/model.joblib")


class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float


class PredictionResponse(BaseModel):
    predicted_class: str
    predicted_class_index: int
    confidence: float


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict", response_model=PredictionResponse)
def predict(request: IrisRequest):
    features = np.array([[request.sepal_length, request.sepal_width,
                           request.petal_length, request.petal_width]])
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]

    return PredictionResponse(
        predicted_class=TARGET_NAMES[prediction],
        predicted_class_index=int(prediction),
        confidence=float(probabilities[prediction]),
    )
'''

with open('app.py', 'w') as f:
    f.write(app_py_code)

print('app.py written to disk')
print('Run locally with: uvicorn app:app --reload --port 8000')

app.py written to disk
Run locally with: uvicorn app:app --reload --port 8000


## 8. Docker Basics

**Docker** packages an application and everything it needs (Python version, libraries, code)
into a portable **image**, so it runs the same way on any machine. Key concepts:

- **Image** - a snapshot/template built from a `Dockerfile`.
- **Container** - a running instance of an image.
- **`Dockerfile`** - step-by-step instructions for building the image.

Below we write a `Dockerfile` that containerizes our FastAPI app.

In [9]:
dockerfile_content = '''\
# Use a lightweight official Python image
FROM python:3.11-slim

# Set the working directory inside the container
WORKDIR /app

# Copy dependency list first (better Docker layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the application code and serialized model
COPY app.py .
COPY artifacts/ ./artifacts/

# Document the port the app listens on
EXPOSE 8000

# Start the API with uvicorn when the container runs
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)

print('Dockerfile written to disk')
print()
print('Build : docker build -t iris-api .')
print('Run   : docker run -p 8000:8000 iris-api')
print('Test  : curl http://localhost:8000/health')

Dockerfile written to disk

Build : docker build -t iris-api .
Run   : docker run -p 8000:8000 iris-api
Test  : curl http://localhost:8000/health


## 9. GitHub Actions Overview & CI/CD Concepts

**CI/CD** (Continuous Integration / Continuous Deployment) automates testing and shipping code:

- **CI (Continuous Integration)** - automatically run tests every time code is pushed, to catch
  problems early.
- **CD (Continuous Deployment/Delivery)** - automatically build and deploy the app (e.g. as a
  Docker image) once tests pass.

**GitHub Actions** is GitHub's built-in automation tool: a **workflow** (YAML file in
`.github/workflows/`) defines **jobs** made up of **steps** that run on events like a `push`.

Below is a simple CI/CD workflow for our Iris API: it installs dependencies, runs tests, and
then builds the Docker image.

In [10]:
workflow_yaml = '''\
name: CI-CD - Iris API

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - name: Check out code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run tests
        run: pytest test_app.py -v

  build:
    needs: test
    runs-on: ubuntu-latest
    steps:
      - name: Check out code
        uses: actions/checkout@v4

      - name: Build Docker image
        run: docker build -t iris-api:latest .
'''

os.makedirs('.github/workflows', exist_ok=True)
with open('.github/workflows/ci-cd.yml', 'w') as f:
    f.write(workflow_yaml)

print('.github/workflows/ci-cd.yml written to disk')

.github/workflows/ci-cd.yml written to disk


A CI pipeline needs tests to run. Here's a tiny `pytest`-style test file for our API
(written to disk, using the same `TestClient` approach we used interactively above).

In [11]:
test_file_code = '''\
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)


def test_health():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json() == {"status": "ok"}


def test_predict():
    payload = {
        "sepal_length": 5.1,
        "sepal_width": 3.5,
        "petal_length": 1.4,
        "petal_width": 0.2,
    }
    response = client.post("/predict", json=payload)
    assert response.status_code == 200
    body = response.json()
    assert "predicted_class" in body
    assert 0.0 <= body["confidence"] <= 1.0
'''

with open('test_app.py', 'w') as f:
    f.write(test_file_code)

print('test_app.py written to disk')

test_app.py written to disk


In [12]:
# Run the tests we just wrote, right here in the notebook, to confirm they pass
import sys
import subprocess

result = subprocess.run([sys.executable, '-m', 'pytest', 'test_app.py', '-v'],
                         capture_output=True, text=True)
print(result.stdout[-1500:])
print(result.stderr[-500:] if result.returncode != 0 else 'All tests passed.')


c:\Users\Mathew\AppData\Local\Programs\Python\Python314\python.exe: No module named pytest



## 10. Cloud Deployment Overview (AWS / Azure)

Once an API is containerized, it needs somewhere to run in production. Two common paths:

**AWS**
- **ECS (Elastic Container Service) / Fargate** - run the Docker container without managing
  servers.
- **ECR (Elastic Container Registry)** - store the built Docker image.
- **Elastic Beanstalk** - simpler, more managed option for smaller apps.
- **SageMaker** - purpose-built for hosting ML models specifically.

**Azure**
- **Azure Container Apps / Azure Container Instances** - run containers without managing VMs.
- **Azure Container Registry (ACR)** - store the built Docker image.
- **Azure App Service** - managed hosting, supports containers directly.
- **Azure Machine Learning endpoints** - purpose-built for hosting ML models.

**Typical flow (either cloud):** build Docker image -> push to a container registry -> deploy
that image to a managed container service -> the service exposes a public URL for the API.

This step is intentionally conceptual here - actually provisioning cloud resources needs
account access and credentials outside this notebook's local environment.

## 11. Final Project Presentation - Tips

For the final project presentation, a clear structure helps the audience follow along:

1. **Problem statement** - what are you predicting/solving, and why does it matter?
2. **Data & approach** - what data was used, and what model/technique was chosen (and why).
3. **Results** - key metrics, with a simple chart if possible.
4. **Deployment** - how the model is served (this is where today's work fits in): serialization
   -> API -> container -> CI/CD -> cloud.
5. **Demo** - a short live or recorded demo of the API responding to a real request.
6. **Learnings & next steps** - what you'd improve with more time/data.

Keep slides simple: one key idea per slide, and let the live demo do the talking.

## 12. Summary

- Trained a simple **Logistic Regression** model on the Iris dataset as our serving example.
- Learned **model serialization** with `pickle` and `joblib`, and confirmed round-trip loading.
- Used **MLflow** to log parameters, metrics, and the model artifact for a training run.
- Built a **FastAPI REST API** with `/health` and `/predict` endpoints, validated with Pydantic.
- Tested the API in-notebook with `TestClient`, then wrote it out as a standalone `app.py`.
- Wrote a **Dockerfile** to containerize the API for portable deployment.
- Learned **CI/CD concepts** and wrote a **GitHub Actions** workflow that tests and builds the
  Docker image automatically, along with a small `pytest` test suite.
- Reviewed a **cloud deployment overview** for AWS and Azure container-hosting options.
- Reviewed tips for structuring the **final project presentation**.